# 04 - Model Training, Comparison & Uncertainty
We forecast AQI **1, 2 and 3 days ahead** (24/48/72 h) with a *direct* strategy: one model per horizon.

Compared: **Persistence baseline** (tomorrow = today), **Ridge regression**, **Random Forest**, **XGBoost**.

Split (chronological): train <= 2018-12-31, validation to 2019-06-30, test afterwards. The test period includes Diwali 2019 and the 2020 COVID lockdown, so it is a genuinely tough test.

**Uncertainty:** for the chosen model we measure how wrong it was on the validation set as a *ratio* (actual / predicted) and use its 10th and 90th percentiles to build an **80 % prediction interval**. Coverage is then checked on the untouched test set.

In [1]:
import json, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from utils import *

data = pd.read_csv(FEATURE_PATH, index_col=0, parse_dates=True)
feature_cols = json.load(open(MODEL_DIR / "feature_cols.json"))
targets = [f"y_h{h}" for h in HORIZONS]
data = data.dropna(subset=feature_cols + targets)

gap = max(HORIZONS)  # drop the last rows of each split so targets don't peek into the next split
train = data.loc[:TRAIN_END].iloc[:-gap]
val = data.loc[pd.Timestamp(TRAIN_END) + pd.Timedelta(days=1): VAL_END].iloc[:-gap]
test = data.loc[pd.Timestamp(VAL_END) + pd.Timedelta(days=1):]
print("train", len(train), "| val", len(val), "| test", len(test))

print("Split dates:", TRAIN_END, VAL_END, "| data:", data.index.min().date(), "->", data.index.max().date())
if min(len(train), len(val), len(test)) < 30:
    raise ValueError(
        f"A split has too few rows (train={len(train)}, val={len(val)}, test={len(test)}). "
        "Re-run 01c, 02 and 03, restart this kernel, and check the 'Rows per year' table in notebook 03.")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\RAJESHWARI SAHU\\AQI-Forecasting-System\\data\\processed\\jalandhar_features.csv'

In [9]:
print("Split dates:", TRAIN_END, VAL_END)
print("Data range :", data.index.min().date(), "->", data.index.max().date(), "| rows:", len(data))
print("train / val / test rows:", len(train), len(val), len(test))

Split dates: 2024-03-07 2024-07-20


NameError: name 'data' is not defined

In [ ]:
def make_models():
    return {
        "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=10.0)),
        "RandomForest": RandomForestRegressor(n_estimators=300, min_samples_leaf=3,
                                              n_jobs=-1, random_state=42),
        "XGBoost": XGBRegressor(n_estimators=400, learning_rate=0.03, max_depth=4,
                                subsample=0.8, colsample_bytree=0.8, random_state=42),
    }

def metrics(y, p):
    return {"MAE": mean_absolute_error(y, p),
            "RMSE": np.sqrt(mean_squared_error(y, p)),
            "R2": r2_score(y, p)}

val_pred, test_pred, rows = {}, {}, []
for h in HORIZONS:
    y_tr, y_va, y_te = train[f"y_h{h}"], val[f"y_h{h}"], test[f"y_h{h}"]

    # Baseline: tomorrow = today
    for split, frame, y in (("val", val, y_va), ("test", test, y_te)):
        rows.append({"horizon": h, "model": "Persistence", "split": split,
                     **metrics(y, frame["aqi_lag0"])})
    val_pred[(h, "Persistence")] = val["aqi_lag0"].values
    test_pred[(h, "Persistence")] = test["aqi_lag0"].values

    for name, model in make_models().items():
        model.fit(train[feature_cols], y_tr)
        joblib.dump(model, MODEL_DIR / f"{name}_h{h}.joblib")
        pv, pt = model.predict(val[feature_cols]), model.predict(test[feature_cols])
        val_pred[(h, name)], test_pred[(h, name)] = pv, pt
        rows.append({"horizon": h, "model": name, "split": "val", **metrics(y_va, pv)})
        rows.append({"horizon": h, "model": name, "split": "test", **metrics(y_te, pt)})

res = pd.DataFrame(rows)
test_res = res[res.split == "test"].pivot(index="model", columns="horizon", values="MAE").round(1)
print("TEST MAE (lower is better)")
test_res

In [ ]:
# Skill score: % improvement over persistence on the test set (positive = beats baseline)
mae = res[res.split == "test"].pivot(index="model", columns="horizon", values="MAE")
skill = (1 - mae.div(mae.loc["Persistence"], axis=1)) * 100
skill.drop("Persistence").round(1).rename(columns=lambda h: f"Day +{h} (% better)")

## Pick the best model per horizon (using the **validation** set only) and build intervals

In [ ]:
model_names = ["Ridge", "RandomForest", "XGBoost"]
val_mae = res[res.split == "val"].pivot(index="model", columns="horizon", values="MAE")
best = {h: val_mae.loc[model_names, h].idxmin() for h in HORIZONS}
print("Best model per horizon:", best)

interval_info = {}
for h in HORIZONS:
    b = best[h]
    pv = np.clip(val_pred[(h, b)], 1, None)
    log_ratio = np.log(np.clip(val[f"y_h{h}"].values, 1, None) / pv)
    q_low, q_high = np.quantile(log_ratio, [0.10, 0.90])

    pt = np.clip(test_pred[(h, b)], 1, None)
    lo, hi = pt * np.exp(q_low), pt * np.exp(q_high)
    cover = float(np.mean((test[f"y_h{h}"].values >= lo) & (test[f"y_h{h}"].values <= hi)))
    interval_info[h] = {"model": b, "log_q_low": float(q_low), "log_q_high": float(q_high),
                        "test_coverage": cover}
    print(f"h={h}: {b:13s} interval = pred x [{np.exp(q_low):.2f}, {np.exp(q_high):.2f}]  "
          f"-> test coverage {cover:.0%} (target 80%)")

json.dump(interval_info, open(MODEL_DIR / "intervals.json", "w"), indent=2)

If the test coverage is clearly below 80 %, that is an honest finding: the future was harder than the validation period (e.g. COVID lockdown). Mention it in your report rather than hiding it.

In [ ]:
# Save test predictions (long format) - used by notebook 05 for a fair comparison
frames = []
for h in HORIZONS:
    b = best[h]
    pt = np.clip(test_pred[(h, b)], 1, None)
    d = pd.DataFrame({"date": test.index, "horizon": h, "y_true": test[f"y_h{h}"].values})
    for name in ["Persistence"] + model_names:
        d[name] = test_pred[(h, name)]
    d["best_pred"] = pt
    d["lo"] = pt * np.exp(interval_info[h]["log_q_low"])
    d["hi"] = pt * np.exp(interval_info[h]["log_q_high"])
    frames.append(d)
preds = pd.concat(frames)
preds.to_csv(RESULTS_DIR / "test_predictions.csv", index=False)

res.to_csv(RESULTS_DIR / "ml_metrics.csv", index=False)
preds.head()

In [ ]:
h = 1
d = preds[preds.horizon == h].set_index("date")
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(d.index, d["y_true"], color="black", lw=1, label="Actual")
ax.plot(d.index, d["Persistence"], color="grey", lw=0.8, ls="--", label="Persistence")
ax.plot(d.index, d["best_pred"], color="#c0392b", lw=1, label=f"{best[h]} forecast")
ax.fill_between(d.index, d["lo"], d["hi"], color="#c0392b", alpha=0.2, label="80% interval")
ax.set_title("Test period, 24-hour-ahead forecast"); ax.set_ylabel("AQI"); ax.legend()
fig.savefig(FIG_DIR / "forecast_h1_test.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# Error by AQI level - where do the models struggle?
d1 = preds[preds.horizon == 1].copy()
d1["category"] = d1["y_true"].apply(aqi_category)
d1["abs_err"] = (d1["y_true"] - d1["best_pred"]).abs()
d1["abs_err_persist"] = (d1["y_true"] - d1["Persistence"]).abs()
d1.groupby("category")[["abs_err", "abs_err_persist"]].mean().reindex(AQI_LABELS).dropna().round(1)